In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# ==================== 1. 数据准备 ====================
# CIFAR-10的均值和标准差（用于归一化）
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

# 数据增强（对训练集）
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # 随机裁剪
    transforms.RandomHorizontalFlip(),      # 随机水平翻转
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 测试集只做归一化
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 加载数据集（假设cifar10文件已经在当前目录下的 ./data 文件夹中）
# 如果文件在别处，修改root参数
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=test_transform
)

# 创建数据加载器
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"类别数: 10")

# CIFAR-10的类别名称
classes = ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车']

# ==================== 2. 定义ResNet模型 ====================
# 基础残差块（BasicBlock）
class BasicBlock(nn.Module):
    """用于ResNet-18/34的基础残差块"""
    expansion = 1  # 输出通道数扩展倍数
    
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        # 跳跃连接（如果维度不匹配，需要做投影）
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = x  # 保存输入用于跳跃连接
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        # 如果输入输出维度不匹配，对identity做变换
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out += identity  # 残差连接
        out = self.relu(out)
        
        return out


# 瓶颈块（Bottleneck），用于ResNet-50/101/152
class Bottleneck(nn.Module):
    """用于更深网络的瓶颈块"""
    expansion = 4
    
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, 
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        
        out = self.conv3(out)
        out = self.bn3(out)
        
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out += identity
        out = self.relu(out)
        
        return out


# ResNet主类
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64
        
        # 初始卷积层（CIFAR-10图像32x32，用7x7太大，改用3x3）
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # 四个残差阶段
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        
        # 全局平均池化 + 全连接分类头
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        
        # 初始化权重
        self._initialize_weights()
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        
        # 如果维度不匹配，需要下采样
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )
        
        layers = []
        # 第一个残差块（可能改变尺寸）
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        
        # 后续残差块（尺寸不变）
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x


# 创建不同深度的ResNet
def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def ResNet34(num_classes=10):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

def ResNet50(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

def ResNet101(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 23, 3], num_classes)


# ==================== 3. 训练函数 ====================
def train_epoch(model, train_loader, criterion, optimizer, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'Loss': f'{running_loss/(batch_idx+1):.3f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    return 100. * correct / total, running_loss / len(train_loader)


def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    accuracy = 100. * correct / total
    avg_loss = test_loss / len(test_loader)
    
    print(f'测试结果 - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
    return accuracy, avg_loss


# ==================== 4. 主程序 ====================
def main():
    # 创建模型
    print("创建 ResNet-18 模型...")
    model = ResNet18(num_classes=10).to(device)
    
    # 打印模型参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"总参数量: {total_params:,}")
    print(f"可训练参数量: {trainable_params:,}")
    
    # 损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    
    # 学习率调度器（每20个epoch衰减到0.1倍）
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 40, 60], gamma=0.1)
    
    # 训练参数
    num_epochs = 50
    best_acc = 0.0
    
    # 记录训练历史
    train_accs = []
    test_accs = []
    
    print("\n开始训练...")
    for epoch in range(1, num_epochs + 1):
        train_acc, train_loss = train_epoch(model, train_loader, criterion, optimizer, epoch)
        test_acc, test_loss = test(model, test_loader, criterion)
        
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        scheduler.step()
        
        # 保存最佳模型
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'best_resnet18_cifar10.pth')
            print(f"✓ 保存最佳模型，准确率: {best_acc:.2f}%")
    
    print(f"\n训练完成！最佳测试准确率: {best_acc:.2f}%")
    
    # 绘制训练曲线
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_accs, label='训练准确率')
    plt.plot(test_accs, label='测试准确率')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.title('训练 vs 测试准确率')
    
    plt.subplot(1, 2, 2)
    # 简单展示几张测试图片的预测结果
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images[:8].to(device), labels[:8]
    outputs = model(images)
    _, preds = outputs.max(1)
    
    # 反归一化用于显示
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    images_display = images.cpu() * std + mean
    images_display = torch.clamp(images_display, 0, 1)
    
    for i in range(8):
        plt.subplot(2, 4, i+1)
        img = images_display[i].permute(1, 2, 0).numpy()
        plt.imshow(img)
        color = 'green' if preds[i] == labels[i] else 'red'
        plt.title(f'真:{classes[labels[i]]}\n预:{classes[preds[i]]}', color=color, fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    plt.show()
    
    print("\n结果图已保存为 training_results.png")


if __name__ == '__main__':
    main()

使用设备: cuda


c:\Users\momo\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


训练集大小: 50000
测试集大小: 10000
类别数: 10
创建 ResNet-18 模型...
总参数量: 11,173,962
可训练参数量: 11,173,962

开始训练...


Epoch 1: 100%|██████████| 391/391 [00:31<00:00, 12.48it/s, Loss=1.963, Acc=29.86%]


测试结果 - Loss: 1.5330, Accuracy: 41.86%
✓ 保存最佳模型，准确率: 41.86%


...

测试结果 - Loss: 0.5411, Accuracy: 81.79%
✓ 保存最佳模型，准确率: 81.79%


Epoch 12: 100%|██████████| 391/391 [00:30<00:00, 12.79it/s, Loss=0.466, Acc=84.34%]


测试结果 - Loss: 0.5321, Accuracy: 82.03%
✓ 保存最佳模型，准确率: 82.03%


Epoch 13: 100%|██████████| 391/391 [00:30<00:00, 12.64it/s, Loss=0.455, Acc=84.36%]


KeyboardInterrupt: 

In [1]:
# ==================== 迁移学习对比实验 ====================
def transfer_learning_comparison():
    """对比：从头训练 vs 使用ImageNet预训练权重"""
    
    from torchvision.models import resnet18
    
    # 方法1：从头训练（上面的代码已经做了）
    
    # 方法2：使用预训练权重（只训练最后几层）
    print("\n" + "="*50)
    print("迁移学习实验：使用ImageNet预训练权重")
    print("="*50)
    
    # 加载预训练模型
    pretrained_model = resnet18(weights='IMAGENET1K_V1')
    
    # 冻结所有层
    for param in pretrained_model.parameters():
        param.requires_grad = False
    
    # 替换最后的全连接层
    num_features = pretrained_model.fc.in_features
    pretrained_model.fc = nn.Linear(num_features, 10)
    
    # 只训练新加的分类头
    pretrained_model = pretrained_model.to(device)
    
    # 使用更小的学习率
    optimizer = optim.SGD(pretrained_model.fc.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    
    # 快速训练10个epoch
    print("快速训练10个epoch（只训练分类头）...")
    for epoch in range(1, 11):
        train_acc, _ = train_epoch(pretrained_model, train_loader, criterion, optimizer, epoch)
        test_acc, _ = test(pretrained_model, test_loader, criterion)
        print(f"Epoch {epoch}: Test Acc = {test_acc:.2f}%")
    
    # 结论：即使只训练分类头，预训练模型通常也能达到比从头训练更高的准确率

In [2]:
def visualize_activations(model, image, layer_name='layer1'):
    """可视化指定层的激活图"""
    activations = []
    
    def hook_fn(module, input, output):
        activations.append(output.detach())
    
    # 注册钩子
    hook = getattr(model, layer_name).register_forward_hook(hook_fn)
    
    model.eval()
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(device))
    
    hook.remove()
    
    # 取前16个通道可视化
    act = activations[0][0]  # [C, H, W]
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for i, ax in enumerate(axes.flat):
        if i < act.shape[0]:
            ax.imshow(act[i].cpu(), cmap='viridis')
            ax.axis('off')
            ax.set_title(f'通道 {i}')
    plt.tight_layout()
    plt.savefig('layer_activations.png')
    plt.show()